In [7]:
%pip install ollama transformers sounddevice

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import ollama

LLM_ID = "qwen3:1.7b"

ollama.chat(model=LLM_ID, messages=[{"role": "user", "content": "hi"}], think=False)
print(f"{LLM_ID} is ready")

qwen3:1.7b is ready


In [9]:
SYSTEM_PROMPT = (
    "You are a helpful voice assistant on smart glasses. "
    "The user's words come from speech recognition and may contain mistakes; guess what they meant. "
    "Answer the user's latest message directly, in one or two short sentences."
)

history = [{"role": "system", "content": SYSTEM_PROMPT}]
MAX_TURNS = 6  # keep the last N question/answer pairs so the prompt doesn't grow forever


def ask_llm(transcript, max_new_tokens=150):
    """Send one ASR transcript to the LLM, stream the answer to stdout and return it."""
    history.append({"role": "user", "content": transcript})
    del history[1:-(2 * MAX_TURNS + 1)]  # drop old turns, keep the system prompt

    answer = ""
    print("LLM:", end=" ", flush=True)
    for part in ollama.chat(
        model=LLM_ID,
        messages=history,
        stream=True,
        think=False,
        options={"temperature": 0.7, "num_predict": max_new_tokens},
    ):
        print(part.message.content, end="", flush=True)
        answer += part.message.content
    print()

    history.append({"role": "assistant", "content": answer.strip()})
    return answer.strip()

In [10]:
history = [{"role": "system", "content": SYSTEM_PROMPT}]

ask_llm("hello my name is ivan what is your name")
ask_llm("what is the capital of france")
ask_llm("and how many people live there")  # follow-up: needs the history to know "there" = Paris

LLM: Hello, Ivan! I'm called AI, your virtual assistant. How can I help you today?
LLM: The capital of France is Paris.
LLM: Paris has approximately 2 million residents.


'Paris has approximately 2 million residents.'

In [11]:
from asr_stream import LiveTranscriber

asr = LiveTranscriber(mic="Headset CMF WASAPI")

Loading weights: 100%|██████████| 651/651 [00:04<00:00, 156.25it/s]


Streaming latency: 560 ms


In [12]:
history = [{"role": "system", "content": SYSTEM_PROMPT}]  # start a fresh conversation

try:
    for transcript in asr.utterances(pause_seconds=1.5):
        ask_llm(transcript)
        print()
except KeyboardInterrupt:
    print("\nStopped.")

Listening... speak now (press Interrupt to stop)
Hello, my name is Ivan. What is your name
LLM: I'm called AI, but you can call me whatever you want! 😊

Okay, could you tell me what is the capital of Bulgaria
LLM: The capital of Bulgaria is Sofia.

Okay, uh, could you tell me
LLM: Sure! What would you like to know?

what is velocity in physics?
LLM: Velocity is a vector quantity that describes both the speed and direction of an object's movement. It is calculated as the change in position divided by the change in time.

Could you give me any uh formula about the velocity how to define it how to
LLM: Velocity is defined as the rate of change of position with respect to time. The formula is:  
$$
v = \frac{\Delta x}{\Delta t}
$$  
Where $ v $ is velocity, $ \Delta x $ is the change in position, and $ \Delta t $ is the change in time.

uh know what is the
LLM: Sure! Let me know if you'd like to learn more about velocity or any other physics concepts.

uh
LLM: Sure! Let me know if you'd li